# Import libraries

In [ ]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import gc
import os

# Read data

In [ ]:
train_df = pd.read_csv('../input/ventilator-pressure-prediction/train.csv')
print(f"train_df: {train_df.shape}")
train_df.head()

# Custom function

In [ ]:
def better_than_median(inputs, axis):
    """Compute the mean of the predictions if there are no outliers,
    or the median if there are outliers.

    Parameter: inputs = ndarray of shape (n_samples, n_folds)"""
    spread = inputs.max(axis=axis) - inputs.min(axis=axis) 
    spread_lim = 0.45
    print(f"Inliers:  {(spread < spread_lim).sum():7} -> compute mean")
    print(f"Outliers: {(spread >= spread_lim).sum():7} -> compute median")
    print(f"Total:    {len(inputs):7}")
    return np.where(spread < spread_lim,
                    np.mean(inputs, axis=axis),
                    np.median(inputs, axis=axis))

# Pressure steps

In [ ]:
targets = train_df[['pressure']].to_numpy().reshape(-1, 80)


pressure = targets.squeeze().reshape(-1,1).astype('float32')

P_MIN = np.min(pressure)
P_MAX = np.max(pressure)
P_STEP = (pressure[1] - pressure[0])[0]
print('Min pressure: {}'.format(P_MIN))
print('Max pressure: {}'.format(P_MAX))
print('Pressure step: {}'.format(P_STEP))
print('Unique values:  {}'.format(np.unique(pressure).shape[0]))

del pressure
_ = gc.collect()

# Ensemble Predictions

In [ ]:
files = [
#     '/kaggle/input/gbvppillifictionfoldpreds/IlliFiction_fold1_submission.csv',
#     '/kaggle/input/gbvppillifictionfoldpreds/IlliFiction_fold2_submission.csv',
#     '/kaggle/input/gbvppillifictionfoldpreds/IlliFiction_fold3_submission.csv',
#     '/kaggle/input/gbvppillifictionfoldpreds/IlliFiction_fold4_submission.csv',
#     '/kaggle/input/gbvppillifictionfoldpreds/IlliFiction_fold5_submission.csv',
#     '/kaggle/input/gbvppillifictionfoldpreds/IlliFiction_fold6_submission.csv',
#     '/kaggle/input/gbvppillifictionfoldpreds/IlliFiction_fold7_submission.csv',
#     '/kaggle/input/gbvppillifictionfoldpreds/IlliFiction_fold8_submission.csv',
#     '/kaggle/input/gbvppillifictionfoldpreds/IlliFiction_fold9_submission.csv',
#     '/kaggle/input/gbvppillifictionfoldpreds/IlliFiction_fold10_submission.csv',
#     '/kaggle/input/gbvpp-foldpreds/IlliFictionV8_fold1_submission.csv',
#     '/kaggle/input/gbvpp-foldpreds/IlliFictionV8_fold2_submission.csv',
#     '/kaggle/input/gbvpp-foldpreds/IlliFictionV8_fold3_submission.csv',
#     '/kaggle/input/gbvpp-foldpreds/IlliFictionV8_fold4_submission.csv',
#     '/kaggle/input/gbvpp-foldpreds/IlliFictionV8_fold5_submission.csv',
#     '/kaggle/input/gbvpp-foldpreds/IlliFictionV8_fold6_submission.csv',
#     '/kaggle/input/gbvpp-foldpreds/IlliFictionV8_fold7_submission.csv',
#     '/kaggle/input/gbvpp-foldpreds/IlliFictionV8_fold8_submission.csv',
#     '/kaggle/input/gbvpp-foldpreds/IlliFictionV8_fold9_submission.csv',
#     '/kaggle/input/gbvpp-foldpreds/BiLSTMSkipNet_fold1_submission.csv',
#     '/kaggle/input/gbvpp-foldpreds/BiLSTMSkipNet_fold2_submission.csv',
#     '/kaggle/input/gbvpp-foldpreds/BiLSTMSkipNet_fold3_submission.csv',
#     '/kaggle/input/gbvpp-foldpreds/BiLSTMSkipNet_fold4_submission.csv',
#     '/kaggle/input/gbvpp-foldpreds/BiLSTMSkipNet_fold5_submission.csv',
#     '/kaggle/input/gbvpp-foldpreds/BiLSTMSkipNet_fold6_submission.csv',
#     '/kaggle/input/gbvpp-foldpreds/BiLSTMSkipNet_fold7_submission.csv',
#     '/kaggle/input/gbvpp-foldpreds/BiLSTMSkipNet_fold8_submission.csv',
#     '/kaggle/input/gbvpp-foldpreds/BiLSTMSkipNet_fold9_submission.csv',
#     '/kaggle/input/gbvpp-foldpreds/BiLSTMSkipNet_fold10_submission.csv',
#     '/kaggle/input/gbvpp-foldpreds/IlliFictionV8_fold10_submission.csv',
    '/kaggle/input/gbvpp-illidub/median_submission.csv',
    '/kaggle/input/gbvpp-submit/IlliFictionV8_median_submission.csv',
    '/kaggle/input/gbvpp-submit/IlliFiction1_median_submission.csv',
    '/kaggle/input/vpp-a-basic-ensembling-technique-with-ridgecv/submission_pp.csv',
    '/kaggle/input/gb-vpp-pulp-fiction/median_submission.csv',
    '/kaggle/input/gb-vpp-pulp-fiction-abb651/median_submission.csv',
    '/kaggle/input/gb-vpp-whoppity-dub-dub/median_submission.csv',
    '/kaggle/input/dnn-lstm-tpu/median_submission.csv',
    '/kaggle/input/ventilator-train-classification/exp080_conti_rc/submission_median_pp.csv'
]

subLen = len(pd.read_csv("/kaggle/input/ventilator-pressure-prediction/sample_submission.csv"))

preds = np.zeros((subLen,len(files)))

preds.shape

In [ ]:
for ind, file in tqdm(enumerate(files), total=len(files)):
    
    df = pd.read_csv(file)
    pred = df[['pressure']].values.squeeze(-1)
    pred = np.round((pred - P_MIN)/P_STEP) * P_STEP + P_MIN
    pred = np.clip(pred, P_MIN, P_MAX)
    preds[:subLen,ind] = pred

# Submission

In [ ]:
sub = pd.read_csv('../input/ventilator-pressure-prediction/sample_submission.csv')
sub['pressure'] = better_than_median(preds, 1)
sub["pressure"] = np.round((sub.pressure - P_MIN)/P_STEP) * P_STEP + P_MIN
sub["pressure"] = np.clip(sub.pressure, P_MIN, P_MAX)
sub.to_csv('final_submission.csv', index=False)
sub.head(5)

In [ ]:
sub.pressure.nunique()